# 03 — Candidate Bicycle Segments

Builds the analysis universe of existing **on-street, non-protected bicycle-route segments** and attaches authoritative LION geometry.

A `SegmentID` is excluded if any on-street bike-route record for that segment is already protected, curbside, or curbside buffered. This avoids retaining a segment merely because a duplicate record describes another side/direction as non-protected.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

project = Path.cwd().resolve()
if project.name == "notebooks":
    project = project.parent

raw = project / "data" / "raw"
interim = project / "data" / "interim"
processed = project / "data" / "processed"
tables_dir = project / "outputs" / "tables"

interim.mkdir(parents=True, exist_ok=True)
processed.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project)

import geopandas as gpd

In [ ]:
bike = gpd.read_file(processed / "bike_routes_2263_nyc.gpkg")
lion = gpd.read_file(processed / "lion_2263.gpkg")

candidate_types = [
    "Conventional",
    "Conventional Buffered",
    "Shared",
    "Signed Route",
    "Wide Parking Lane"
]

protected_types = [
    "Protected",
    "Curbside",
    "Curbside Buffered"
]

bike_on = bike[bike["onoffst"] == "ON"].copy()

candidate_ids_all = set(
    bike_on.loc[
        bike_on["ft_facilit"].isin(candidate_types) |
        bike_on["tf_facilit"].isin(candidate_types),
        "segmentid"
    ]
)

protected_ids_all = set(
    bike_on.loc[
        bike_on["ft_facilit"].isin(protected_types) |
        bike_on["tf_facilit"].isin(protected_types),
        "segmentid"
    ]
)

final_candidate_ids = candidate_ids_all - protected_ids_all

print("Candidate IDs before protected exclusion:", len(candidate_ids_all))
print("Protected IDs removed:", len(candidate_ids_all & protected_ids_all))
print("Final non-protected candidate IDs:", len(final_candidate_ids))

In [ ]:
bike_candidates = bike_on[
    bike_on["segmentid"].isin(final_candidate_ids)
].copy()

bike_segment_summary = (
    bike_candidates
    .groupby("segmentid")
    .agg(
        facility_classes=(
            "facilitycl",
            lambda x: ",".join(sorted(set(x.dropna().astype(str))))
        ),
        allclasses=(
            "allclasses",
            lambda x: ",".join(sorted(set(x.dropna().astype(str))))
        ),
        ft_facilities=(
            "ft_facilit",
            lambda x: ",".join(sorted(set(x.dropna().astype(str))))
        ),
        tf_facilities=(
            "tf_facilit",
            lambda x: ",".join(sorted(set(x.dropna().astype(str))))
        ),
        route_records=("segmentid", "size")
    )
    .reset_index()
)

bike_segment_summary["SegmentID"] = pd.to_numeric(
    bike_segment_summary["segmentid"],
    errors="coerce"
).astype("Int64")

lion["SegmentID"] = pd.to_numeric(
    lion["SegmentID"],
    errors="coerce"
).astype("Int64")

## LION duplicate QA

In [ ]:
candidate_ids = set(bike_segment_summary["SegmentID"].dropna())
lion_candidates = lion[lion["SegmentID"].isin(candidate_ids)].copy()

lion_candidates["geom_wkb"] = lion_candidates.geometry.apply(
    lambda g: g.wkb_hex if g is not None else None
)

geom_check = (
    lion_candidates
    .groupby("SegmentID")
    .agg(
        rows=("SegmentID", "size"),
        unique_geometries=("geom_wkb", "nunique"),
        unique_physicalids=("PhysicalID", "nunique")
    )
    .reset_index()
)

print("Candidate SegmentIDs with multiple LION rows:",
      (geom_check["rows"] > 1).sum())
print("With multiple different geometries:",
      (geom_check["unique_geometries"] > 1).sum())
print("With multiple PhysicalIDs:",
      (geom_check["unique_physicalids"] > 1).sum())

In [ ]:
lion_unique = (
    lion
    .sort_values("SegmentID")
    .drop_duplicates(subset="SegmentID", keep="first")
    .copy()
)

bike_candidates_lion = lion_unique.merge(
    bike_segment_summary,
    on="SegmentID",
    how="inner"
)

missing_final_ids = sorted(
    set(bike_segment_summary["SegmentID"].dropna()) -
    set(lion_unique["SegmentID"].dropna())
)

print("Final candidate IDs missing from LION:", len(missing_final_ids))
print(missing_final_ids)
print("Usable candidate rows:", len(bike_candidates_lion))

In [ ]:
# geopackage field names are case-insensitive; remove the redundant lowercase ID.
bike_candidates_lion = bike_candidates_lion.drop(
    columns=["segmentid"],
    errors="ignore"
)

candidate_out = processed / "bike_candidates_nonprotected_2263.gpkg"
bike_candidates_lion.to_file(candidate_out, driver="GPKG")

print("Saved:", candidate_out)
print("Rows:", len(bike_candidates_lion))
print("Unique SegmentIDs:", bike_candidates_lion["SegmentID"].nunique())